# DBRepo Normalisation + Clean CSV Export Pipeline
**Project:** Predicting the Market Value of Football Players  

Loads both raw Excel datasets, normalises into 3NF tables matching `schema.sql`,
and exports clean CSVs ready for upload via the DBRepo UI.

## Issues fixed vs previous version
| Issue | Fix |
|---|---|
| `position` column named `position` not `position_name` | Renamed to match schema |
| `nationality` column named `nationality` not `nationality_name` | Renamed to match schema |
| `source_dataset` only had 2 columns | Now has all 11 schema columns |
| `forward_valuation_id` included in CSV | Removed — DB auto-generates it |
| `transfer_observation_id` included in CSV | Removed — DB auto-generates it |
| `club_performance`/`success_or_not` exported as `-1.0` float | Converted to integer `-1` |
| `relegation` exported as `1.0`/`0.0` float | Converted to integer `1`/`0` (BOOLEAN) |
| `start_value_eur`/`end_value_eur` nulls filled with `0.0` | Left as empty cell (true NULL) |
| `encoding='utf-8-sig'` adds BOM | Changed to plain `utf-8` |
| Windows line endings `\r\n` | Fixed with `lineterminator='\n'` |
| `plays_in_europe` not explicitly cast | Cast to int (1/0) for BOOLEAN column |

## Upload order (respect FK dependencies)
```
1. source_dataset  2. player  3. club  4. position  5. nationality  6. season*
7. forward_player_valuation   8. transfer_value_observation
```
*season rows are pre-inserted by schema.sql — do NOT upload season.csv

## 0. Imports and Configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR      = Path("../data")             # folder with the two Excel files
FORWARD_FILE  = DATA_DIR / "Dataset soccerplayers.xlsx"
TRANSFER_FILE = DATA_DIR / "Nisanov_final_data.xlsx"

EXPORT_DIR = Path("../normalized_exports")
EXPORT_DIR.mkdir(exist_ok=True)

print(f"Forward file  exists: {FORWARD_FILE.exists()}")
print(f"Transfer file exists: {TRANSFER_FILE.exists()}")
print(f"Export dir: {EXPORT_DIR.resolve()}")

Forward file  exists: True
Transfer file exists: True
Export dir: C:\Users\USER\Documents\mastersInCompSc\2026s\dataStewardsUe Reports\group assignment main\fair-ds-experiment\normalized_exports


## 1. Load Raw Data

In [2]:
# ── Dataset 1: Forward player valuation ───────────────────────────────────────
forward_df = pd.read_excel(FORWARD_FILE)
forward_df.columns = [c.strip().lower() for c in forward_df.columns]
print(f"Dataset 1: {forward_df.shape}")

# ── Dataset 2: Transfer Value Determinants (5 sheets 2019-2023) ───────────────
transfer_sheets = pd.read_excel(TRANSFER_FILE, sheet_name=None)
transfer_df = pd.concat(
    [sheet.dropna(axis=1, how='all') for sheet in transfer_sheets.values()],
    ignore_index=True,
    sort=False,
)
transfer_df.columns = [c.strip().lower() for c in transfer_df.columns]
print(f"Dataset 2: {transfer_df.shape}")

Dataset 1: (438, 12)
Dataset 2: (2502, 22)


## 2. Fix Data Quality Issues

In [3]:
# ── Fix 1: Garbled player name (UTF-8 encoding corruption) ────────────────────
# '脙聡aglar Soyuncu' appears only in the 2019 sheet.
# All other rows for this player use 'caglar Soyuncu' so we normalise to that.
garbled_mask = transfer_df['player_name'] == '脙聡aglar Soyuncu'
print(f"Garbled name rows: {garbled_mask.sum()}")
transfer_df.loc[garbled_mask, 'player_name'] = 'caglar Soyuncu'

# ── Fix 2: instagram_followers_mln — 6 NaN rows ───────────────────────────────
# These players have no public Instagram. Fill with 0.0 (zero followers).
# We do this later after building forward_fact.

# ── Fix 3: Nullable columns in transfer ───────────────────────────────────────
# club_performance (1157 NaN), relegation (1157), success_or_not (1139)
# start_value_eur (17 NaN), end_value_eur (2 NaN)
# These are genuinely NULL — only the 2019 sheet has them.
# We keep NaN and export as empty cell → DBRepo treats as NULL.
print("Nullable column null counts (correct):")
for col in ['club_performance','relegation','success_or_not','start_value','end_value']:
    print(f"  {col}: {transfer_df[col].isna().sum()}")

Garbled name rows: 1
Nullable column null counts (correct):
  club_performance: 1157
  relegation: 1157
  success_or_not: 1139
  start_value: 17
  end_value: 2


## 3. Build Lookup Tables

In [4]:
# ── player ────────────────────────────────────────────────────────────────────
# Combined from both datasets, deduplicated, sorted
player_names = pd.concat([
    forward_df['nombre'].rename('player_name'),
    transfer_df['player_name'],
]).dropna().drop_duplicates().sort_values().reset_index(drop=True)

player_df = pd.DataFrame({
    'player_id':   range(1, len(player_names) + 1),
    'player_name': player_names.values,
})
print(f"player:      {len(player_df)} rows")

# ── club ──────────────────────────────────────────────────────────────────────
club_names = pd.concat([
    forward_df['team'].rename('club_name'),
    transfer_df['club_name'],
]).dropna().drop_duplicates().sort_values().reset_index(drop=True)

club_df = pd.DataFrame({
    'club_id':   range(1, len(club_names) + 1),
    'club_name': club_names.values,
})
print(f"club:        {len(club_df)} rows")

# ── position ──────────────────────────────────────────────────────────────────
# IMPORTANT: column must be named 'position_name' to match schema.sql
# Previous version incorrectly named it 'position'
position_names = (
    transfer_df['position']
    .dropna().drop_duplicates().sort_values().reset_index(drop=True)
)
position_df = pd.DataFrame({
    'position_id':   range(1, len(position_names) + 1),
    'position_name': position_names.values,   # correct column name
})
print(f"position:    {len(position_df)} rows")

# ── nationality ───────────────────────────────────────────────────────────────
# IMPORTANT: column must be named 'nationality_name' to match schema.sql
# Previous version incorrectly named it 'nationality'
nationality_names = (
    transfer_df['nationality']
    .dropna().drop_duplicates().sort_values().reset_index(drop=True)
)
nationality_df = pd.DataFrame({
    'nationality_id':   range(1, len(nationality_names) + 1),
    'nationality_name': nationality_names.values,   # correct column name
})
print(f"nationality: {len(nationality_df)} rows")

player:      996 rows
club:        205 rows
position:    13 rows
nationality: 75 rows


## 4. Build `source_dataset` Table
Must have all 11 columns from schema.sql. Not just 2 like the previous version.

In [5]:
source_dataset_df = pd.DataFrame([
    {
        'source_dataset_id': 1,
        'dataset_title':     'Forward football player valuation',
        'original_creators': 'Hugo Briseño; José Carlos Rivera',
        'publisher':         'Mendeley Data',
        'version_label':     'V1',
        'doi':               '10.17632/cgc33scxg7.1',
        'doi_url':           'https://doi.org/10.17632/cgc33scxg7.1',
        'license_name':      'CC BY 4.0',
        'license_url':       'https://creativecommons.org/licenses/by/4.0/',
        'source_filename':   'soccerplayers.xlsx',
        'notes':             'Reused external source dataset. Not original publisher.',
    },
    {
        'source_dataset_id': 2,
        'dataset_title':     'Transfer Value Determinants',
        'original_creators': 'Ronald Nisanov',
        'publisher':         'Mendeley Data',
        'version_label':     'V2',
        'doi':               '10.17632/3btg6ptc7b.2',
        'doi_url':           'https://doi.org/10.17632/3btg6ptc7b.2',
        'license_name':      'CC BY 4.0',
        'license_url':       'https://creativecommons.org/licenses/by/4.0/',
        'source_filename':   'Nisanov_final_data.xlsx',
        'notes':             'Reused external source dataset. Not original publisher.',
    },
])
print(f"source_dataset: {len(source_dataset_df)} rows, {len(source_dataset_df.columns)} columns")

source_dataset: 2 rows, 11 columns


## 5. Build `forward_player_valuation` Fact Table

**Key decisions:**
- `forward_valuation_id` is NOT included — the DB generates it via AUTO INCREMENT
- `plays_in_europe` is BOOLEAN in schema → export as integer `1`/`0`
- `instagram_followers_mln` is DECIMAL(12,3) NULL → fill 6 NaN with `0.0`

In [6]:
fwd = forward_df.copy()

# Join lookup IDs
fwd = fwd.merge(
    player_df.rename(columns={'player_name': 'nombre'}),
    on='nombre', how='left',
)
fwd = fwd.merge(
    club_df.rename(columns={'club_name': 'team'}),
    on='team', how='left',
)

# Verify all rows matched
assert fwd['player_id'].isna().sum() == 0, "Unmatched players"
assert fwd['club_id'].isna().sum() == 0,   "Unmatched clubs"

forward_fact = pd.DataFrame({
    # NO forward_valuation_id — auto-generated by DB, must not be in CSV
    'source_dataset_id':       1,
    'player_id':               fwd['player_id'].astype(int),
    'club_id':                 fwd['club_id'].astype(int),
    'original_player_name':    fwd['nombre'],
    'original_team_name':      fwd['team'],
    'player_age_years':        fwd['age'].astype(int),         # SMALLINT
    'market_value_mln':        fwd['value'].round(3),          # DECIMAL(12,3)
    'value_rank':              fwd['value_r'].astype(int),     # SMALLINT
    # BOOLEAN in schema — use integer 1/0 which all DB systems accept for BOOLEAN
    'plays_in_europe':         fwd['europa'].astype(int),
    'matches_played':          fwd['matches'].astype(int),     # INT
    'goals':                   fwd['goals'].astype(int),       # INT
    'assists':                 fwd['assists'].astype(int),     # INT
    'minutes_per_goal':        fwd['mpg'].astype(int),         # INT
    'minutes_played':          fwd['minutes'].astype(int),     # INT
    # DECIMAL(12,3) NULL — fill 6 NaN with 0.0 (player has no Instagram)
    'instagram_followers_mln': fwd['insta'].fillna(0.0).round(3),
})

# Final check
nulls = forward_fact.isnull().sum()
print(f"forward_player_valuation: {len(forward_fact)} rows x {len(forward_fact.columns)} cols")
print(f"  Nulls remaining: {nulls[nulls>0].to_dict() or 'none'}")
print(f"  plays_in_europe values: {forward_fact['plays_in_europe'].unique().tolist()}")
print(f"  market_value_mln sample: {forward_fact['market_value_mln'].head(3).tolist()}")

forward_player_valuation: 438 rows x 15 cols
  Nulls remaining: none
  plays_in_europe values: [1, 0]
  market_value_mln sample: [40.0, 40.0, 40.0]


## 6. Build `transfer_value_observation` Fact Table

**Key decisions:**
- `transfer_observation_id` is NOT included — auto-generated by DB
- `club_performance` → SMALLINT NULL: integers `-1,0,1...` or empty cell
- `relegation` → BOOLEAN NULL: integer `1`/`0` or empty cell
- `success_or_not` → SMALLINT NULL: integers `-1,0,1` or empty cell
- `start_value_eur`, `end_value_eur` → DECIMAL NULL: keep as empty cell (true NULL)
  Previous version incorrectly filled these with `0.0` which is wrong data

In [7]:
trf = transfer_df.copy()

# Join all lookup IDs
trf = trf.merge(player_df, on='player_name', how='left')
trf = trf.merge(club_df,   on='club_name',   how='left')
trf = trf.merge(
    position_df.rename(columns={'position_name': 'position'}),
    on='position', how='left',
)
trf = trf.merge(
    nationality_df.rename(columns={'nationality_name': 'nationality'}),
    on='nationality', how='left',
)

# Verify all FK rows matched
assert trf['player_id'].isna().sum() == 0,      "Unmatched players"
assert trf['club_id'].isna().sum() == 0,         "Unmatched clubs"
assert trf['position_id'].isna().sum() == 0,     "Unmatched positions"
assert trf['nationality_id'].isna().sum() == 0,  "Unmatched nationalities"

# ── Convert nullable integer columns ──────────────────────────────────────────
# club_performance and success_or_not are SMALLINT NULL.
# Raw data has them as float64 because NaN forces pandas to use float.
# pd.Int64Dtype() (capital I) is pandas nullable integer — NaN stays as <NA>
# which exports as empty string '' in CSV = NULL in DB.
# Non-null values export as clean integers (-1, 0, 1) not floats (-1.0)
club_perf   = trf['club_performance'].astype(pd.Int64Dtype())
success     = trf['success_or_not'].astype(pd.Int64Dtype())

# ── Convert nullable boolean column ───────────────────────────────────────────
# relegation is BOOLEAN NULL.
# Map 1.0->1, 0.0->0, NaN stays NaN then convert to Int64 for clean 1/0 export
relegation  = trf['relegation'].map({1.0: 1, 0.0: 0}).astype(pd.Int64Dtype())

transfer_fact = pd.DataFrame({
    # NO transfer_observation_id — auto-generated by DB
    'source_dataset_id':         2,
    'player_id':                 trf['player_id'].astype(int),
    'position_id':               trf['position_id'].astype(int),
    'nationality_id':            trf['nationality_id'].astype(int),
    'club_id':                   trf['club_id'].astype(int),
    'season_year':               trf['season'].astype(int),           # SMALLINT FK
    'original_player_name':      trf['player_name'],
    'original_position_name':    trf['position'],
    'original_nationality_name': trf['nationality'],
    'original_club_name':        trf['club_name'],
    'age_then_years':            trf['age_then'].astype(int),         # SMALLINT
    'age_now_years':             trf['age_now'].astype(int),          # SMALLINT
    # SMALLINT NULL — clean integers or empty cell
    'club_performance':          club_perf,
    # BOOLEAN NULL — 1/0 integer or empty cell
    'relegation':                relegation,
    # SMALLINT NULL — clean integers or empty cell
    'success_or_not':            success,
    'total_games':               trf['total_games'].astype(int),      # INT
    'assists':                   trf['assists'].astype(int),          # INT
    'penalty_kicks':             trf['penalty_kicks'].astype(int),    # INT
    'total_minutes':             trf['total_minutes'].astype(int),    # INT
    'total_goals':               trf['total_goals'].astype(int),      # INT
    'height_cm':                 trf['height'].round(2),              # DECIMAL(6,2)
    # DECIMAL(16,2) NULL — keep NaN as empty cell, do NOT fill with 0
    # 0 would be wrong data (player had unknown start/end value, not zero value)
    'start_value_eur':           trf['start_value'].round(2),
    'end_value_eur':             trf['end_value'].round(2),
    'delta_value_eur':           trf['delta_value'].astype(float).round(2), # DECIMAL(16,2)
    'value_start_mln':           trf['value_0_mln'].round(3),         # DECIMAL(12,3)
    'value_end_mln':             trf['value_end_mln'].round(3),       # DECIMAL(12,3) TARGET
    'value_delta_mln':           trf['value_delta_mln'].round(3),     # DECIMAL(12,3)
})

# Show null summary — only nullable columns should appear
nulls = transfer_fact.isnull().sum()
null_cols = nulls[nulls > 0]
print(f"transfer_value_observation: {len(transfer_fact)} rows x {len(transfer_fact.columns)} cols")
print("Nullable columns (will be empty cells in CSV = NULL in DB):")
for col, count in null_cols.items():
    print(f"  {col}: {count} nulls")

transfer_value_observation: 2502 rows x 27 cols
Nullable columns (will be empty cells in CSV = NULL in DB):
  club_performance: 1157 nulls
  relegation: 1157 nulls
  success_or_not: 1139 nulls
  start_value_eur: 17 nulls
  end_value_eur: 2 nulls


## 7. Verify CSV Content Before Export
Spot-check that integer columns are truly integers, not floats.

In [8]:
import io

def check_csv_values(df, col, label):
    """Write to string buffer and read back to check exact CSV representation."""
    buf = io.StringIO()
    df[[col]].head(5).to_csv(buf, index=False, na_rep='', lineterminator='\n')
    buf.seek(0)
    lines = buf.read().strip().split('\n')[1:]  # skip header
    print(f"  {label}: {lines}")

print("Forward fact — spot checks:")
check_csv_values(forward_fact, 'plays_in_europe',    'plays_in_europe (BOOLEAN)')
check_csv_values(forward_fact, 'market_value_mln',   'market_value_mln (DECIMAL)')
check_csv_values(forward_fact, 'player_age_years',   'player_age_years (SMALLINT)')

print("\nTransfer fact — spot checks:")
check_csv_values(transfer_fact, 'club_performance',  'club_performance (SMALLINT NULL)')
check_csv_values(transfer_fact, 'relegation',        'relegation (BOOLEAN NULL)')
check_csv_values(transfer_fact, 'success_or_not',    'success_or_not (SMALLINT NULL)')
check_csv_values(transfer_fact, 'start_value_eur',   'start_value_eur (DECIMAL NULL)')
check_csv_values(transfer_fact, 'height_cm',         'height_cm (DECIMAL)')

# Find a row where club_performance IS null to confirm empty cell
null_idx = transfer_fact[transfer_fact['club_performance'].isna()].index[0]
buf = io.StringIO()
transfer_fact.loc[[null_idx], ['club_performance','relegation','success_or_not']].to_csv(
    buf, index=False, na_rep='', lineterminator='\n'
)
buf.seek(0)
print(f"\nNull row sample (club_performance,relegation,success_or_not):")
print(buf.read())

Forward fact — spot checks:
  plays_in_europe (BOOLEAN): ['1', '1', '1', '1', '1']
  market_value_mln (DECIMAL): ['40.0', '40.0', '40.0', '40.0', '40.0']
  player_age_years (SMALLINT): ['19', '23', '23', '24', '26']

Transfer fact — spot checks:
  club_performance (SMALLINT NULL): ['-1', '-1', '-1', '-1', '-1']
  relegation (BOOLEAN NULL): ['1', '1', '1', '1', '1']
  success_or_not (SMALLINT NULL): ['-1', '-1', '-1', '-1', '-1']
  start_value_eur (DECIMAL NULL): ['25000000.0', '15000000.0', '3500000.0', '2500000.0', '15000000.0']
  height_cm (DECIMAL): ['179', '193', '183', '180', '186']

Null row sample (club_performance,relegation,success_or_not):
club_performance,relegation,success_or_not
,,



## 8. Export All CSVs

**Export settings:**
- `encoding='utf-8'` — plain UTF-8, no BOM (`utf-8-sig` adds a BOM that can confuse parsers)
- `lineterminator='\n'` — Unix line endings. Without this, Windows Python defaults to `\r\n`
  which some server-side parsers mishandle
- `na_rep=''` — NaN becomes empty cell in CSV = NULL in database
- `index=False` — no row numbers

In [9]:
import csv
def export_csv(df, filename):
    """Export DataFrame to CSV with DBRepo-safe settings."""
    path = EXPORT_DIR / filename
    df.to_csv(
        path,
        index=False,
        encoding='utf-8',        # no BOM
        na_rep='',               # NaN -> empty cell -> NULL in DB
        # lineterminator='\n',     # Unix line endings, not \r\n
        lineterminator='\r\n',      # matches UI line_termination
        quoting=csv.QUOTE_ALL,      # matches UI quote: "\""
        quotechar='"',
    )
    # Verify file starts with column name (no BOM byte)
    with open(path, 'rb') as f:
        first_bytes = f.read(3)
    has_bom = first_bytes == b'\xef\xbb\xbf'
    size_kb = path.stat().st_size / 1024
    status = 'BOM!' if has_bom else 'OK'
    print(f"  [{status}] {filename:<45} {len(df):>5} rows  {size_kb:>7.1f} KB")


print("Exporting CSVs...")
print(f"  {'File':<47} {'Rows':>5}  {'Size':>8}")
print("  " + "-" * 62)

# Upload order matters — lookup tables before fact tables
export_csv(source_dataset_df,  'source_dataset.csv')
export_csv(player_df,          'player.csv')
export_csv(club_df,            'club.csv')
export_csv(position_df,        'position.csv')
export_csv(nationality_df,     'nationality.csv')
# NOTE: season.csv is NOT exported — schema.sql pre-inserts 2019-2023 rows
# Uploading season.csv would cause PRIMARY KEY duplicate errors
export_csv(forward_fact,       'forward_player_valuation.csv')
export_csv(transfer_fact,      'transfer_value_observation.csv')

print()
print(f"All CSVs saved to: {EXPORT_DIR.resolve()}")
print()
print("IMPORTANT: Do NOT upload season.csv — those rows are pre-loaded by schema.sql")

Exporting CSVs...
  File                                             Rows      Size
  --------------------------------------------------------------
  [OK] source_dataset.csv                                2 rows      0.7 KB
  [OK] player.csv                                      996 rows     22.5 KB
  [OK] club.csv                                        205 rows      4.0 KB
  [OK] position.csv                                     13 rows      0.3 KB
  [OK] nationality.csv                                  75 rows      1.3 KB
  [OK] forward_player_valuation.csv                    438 rows     43.5 KB
  [OK] transfer_value_observation.csv                 2502 rows    489.7 KB

All CSVs saved to: C:\Users\USER\Documents\mastersInCompSc\2026s\dataStewardsUe Reports\group assignment main\fair-ds-experiment\normalized_exports

IMPORTANT: Do NOT upload season.csv — those rows are pre-loaded by schema.sql


## 9. Final Verification
Re-reads each exported CSV and confirms row counts and column names.

In [10]:
checks = [
    ('source_dataset.csv',             source_dataset_df,  2),
    ('player.csv',                     player_df,          None),
    ('club.csv',                       club_df,            None),
    ('position.csv',                   position_df,        None),
    ('nationality.csv',                nationality_df,     None),
    ('forward_player_valuation.csv',   forward_fact,       438),
    ('transfer_value_observation.csv', transfer_fact,      2502),
]

print("Final verification:")
all_ok = True

for filename, original_df, expected_rows in checks:
    path = EXPORT_DIR / filename
    reloaded = pd.read_csv(path, encoding='utf-8')

    row_ok = (expected_rows is None) or (len(reloaded) == expected_rows)
    col_ok = list(reloaded.columns) == list(original_df.columns)
    ok = row_ok and col_ok
    all_ok = all_ok and ok

    status = 'OK  ' if ok else 'FAIL'
    print(f"  [{status}] {filename}")
    if not row_ok:
        print(f"         rows: expected {expected_rows}, got {len(reloaded)}")
    if not col_ok:
        missing = set(original_df.columns) - set(reloaded.columns)
        extra   = set(reloaded.columns) - set(original_df.columns)
        if missing: print(f"         missing cols: {missing}")
        if extra:   print(f"         extra cols:   {extra}")

print()
if all_ok:
    print("All checks passed. Ready to upload via DBRepo UI.")
else:
    print("Some checks failed — fix before uploading.")

Final verification:
  [OK  ] source_dataset.csv
  [OK  ] player.csv
  [OK  ] club.csv
  [OK  ] position.csv
  [OK  ] nationality.csv
  [OK  ] forward_player_valuation.csv
  [OK  ] transfer_value_observation.csv

All checks passed. Ready to upload via DBRepo UI.
